<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第3章：评估推理模型

In [1]:
%pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt  # 安装项目所需的所有依赖
# 从 Python 标准库的 importlib.metadata 模块中导入 version 函数，用于查询已安装库的版本
from importlib.metadata import version

# 定义一个列表，列出要检查版本号的第三方库名称
used_libraries = [
    "reasoning_from_scratch",  # 本项目的主库
    "torch",                   # PyTorch，用于深度学习
    "sympy",                   # Sympy，用于符号计算
    "tokenizers"               # tokenizers，被 reasoning_from_scratch 依赖
]

# 遍历上面定义的库列表，对每个库输出其版本号
for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")  # 使用 version 函数查询并打印库的版本


reasoning_from_scratch version: 0.1.10
torch version: 2.9.0+cu126
sympy version: 1.14.0
tokenizers version: 0.22.1


<br>

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F01_raschka.webp?1" width="500px">

&nbsp;
## 3.1 构建数学验证器

- 本节没有代码。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F02_raschka.webp?1" width="500px">

<br>
<br>
<br>

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F03_raschka.webp?1" width="500px">

&nbsp;
## 3.2 加载预训练模型进行文本生成

- 本节会再次加载第2章中要评估的那个模型。

In [2]:
from pathlib import Path  # Path 用于跨平台文件路径操作
import torch  # PyTorch 深度学习库

from reasoning_from_scratch.qwen3 import (  # 导入项目中的 Qwen3 相关工具
    download_qwen3_small,                   # 下载小型 Qwen3 模型或分词器
    Qwen3Tokenizer,                         # Qwen3 分词器类
    Qwen3Model,                             # Qwen3 模型类
    QWEN_CONFIG_06_B                        # 0.6B 参数规模的模型配置
)


def load_model_and_tokenizer(  # 加载模型和分词器的函数
    which_model, device, use_compile, local_dir="qwen3"  # 指定模型类型、设备、是否编译和本地存放目录
):
    if which_model == "base":  # 如果选择基础模型

        download_qwen3_small(  # 下载基础版模型权重和分词器
            kind="base", tokenizer_only=False, out_dir=local_dir
        )

        tokenizer_path = Path(local_dir) / "tokenizer-base.json"  # 基础版分词器路径
        model_path = Path(local_dir) / "qwen3-0.6B-base.pth"      # 基础版模型权重路径
        tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)  # 创建基础版分词器实例

    elif which_model == "reasoning":  # 如果选择 reasoning 版模型

        download_qwen3_small(  # 下载 reasoning 版模型权重和分词器
            kind="reasoning", tokenizer_only=False, out_dir=local_dir
        )

        tokenizer_path = Path(local_dir) / "tokenizer-reasoning.json"  # reasoning 版分词器路径
        model_path = Path(local_dir) / "qwen3-0.6B-reasoning.pth"      # reasoning 版模型权重路径
        tokenizer = Qwen3Tokenizer(  # 创建 reasoning 版分词器实例，并启用聊天模板与“思维链”标志
            tokenizer_file_path=tokenizer_path,
            apply_chat_template=True,
            add_generation_prompt=True,
            add_thinking=True,
        )

    else:  # 传入的模型类型不合法时抛出错误
        raise ValueError(f"Invalid choice: which_model={which_model}")

    model = Qwen3Model(QWEN_CONFIG_06_B)  # 根据配置创建模型实例
    model.load_state_dict(torch.load(model_path))  # 加载权重到模型

    model.to(device)  # 将模型移动到指定设备（CPU/GPU）

    if use_compile:  # 如需使用 torch.compile 进行加速
        torch._dynamo.config.allow_unspec_int_on_nn_module = True  # 允许动态形状中的未指定整型
        model = torch.compile(model)  # 编译模型以获得潜在速度提升

    return model, tokenizer  # 返回模型和分词器实例


- 这里我们使用基础版模型；完成本章后，你可以把`which_model="base"` 改成 `which_model="reasoning"` 再跑一次 notebook，以评估已经训练好的推理模型。

In [3]:
from reasoning_from_scratch.ch02 import (  # 从项目 ch02 模块导入工具函数
    get_device                           # 获取可用计算设备（GPU/CPU）
)

WHICH_MODEL = "base"                     # 选择要加载的模型类型
device = get_device()                    # 自动检测并返回设备对象

# 如果遇到兼容性问题，可将下行取消注释强制使用 CPU
# device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(  # 加载模型与分词器
    which_model=WHICH_MODEL,                  # 指定模型类型
    device=device,                            # 放到检测到的设备上
    use_compile=False                         # 是否启用 torch.compile 加速
)


Using NVIDIA CUDA GPU
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


- 这里不用第2章的 `generate_text_basic_stream` ，而是改用稍作修改的`generate_text_basic_stream_cache`  (见 [exercise 2.2](../../ch02/01_main-chapter-code/ch02_exercise-solutions.ipynb) 。它会在 token 生成后立刻打印，方便调试，避免生成过程中看起来像 LLM “卡住” 了。

In [4]:
from reasoning_from_scratch.ch02_ex import (     # 从章节 2 的扩展模块导入文本生成函数
    generate_text_basic_stream_cache             # 基本流式生成（带缓存）
)

prompt = (                                       # 构造数学题提示词（原样字符串）
    r"If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?"
)

# 类似第 2 章练习的做法：将提示词编码为张量并添加 batch 维
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),                    # 分词器编码为 token id 列表
    device=device                                # 放到目标设备
).unsqueeze(0)                                   # 添加 batch 维度 (1, seq_len)

all_token_ids = []                               # 收集生成的 token id
for token in generate_text_basic_stream_cache(   # 逐 token 流式生成
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=2048,                         # 最多生成 2048 个新 token
    eos_token_id=tokenizer.eos_token_id          # 遇到 EOS 即停
):
    token_id = token.squeeze(0)                  # 去掉 batch 维
    decoded_id = tokenizer.decode(token_id.tolist())  # 解码单个 token
    print(
        decoded_id,
        end="",
        flush=True                               # 实时输出到控制台
    )
    all_token_ids.append(token_id)               # 保存 token id

all_tokens = tokenizer.decode(all_token_ids)     # 将全部生成结果解码为字符串


 To find the value of \( a^2 + b^2 \) given that \( a + b = 3 \) and \( ab = \frac{13}{6} \), we can use the following algebraic identity:

\[
a^2 + b^2 = (a + b)^2 - 2ab
\]

**Step 1:** Substitute the given values into the equation.

\[
a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
\]

**Step 2:** Calculate \( (3)^2 \).

\[
(3)^2 = 9
\]

**Step 3:** Calculate \( 2 \times \frac{13}{6} \).

\[
2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3}
\]

**Step 4:** Subtract the second result from the first.

\[
a^2 + b^2 = 9 - \frac{13}{3}
\]

**Step 5:** Convert 9 to a fraction with a denominator of 3 to perform the subtraction.

\[
9 = \frac{27}{3}
\]

\[
a^2 + b^2 = \frac{27}{3} - \frac{13}{3} = \frac{14}{3}
\]

**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]

- 如果你不熟悉 LaTeX 语法，上面的回答会很难读。
- 可以像下面这样用`Latex` 类把 LaTeX 渲染成更易读的格式。

In [5]:
from IPython.display import Latex, display  # 导入 IPython 显示工具，用于渲染 LaTeX

display(Latex(all_tokens))                  # 将生成文本作为 LaTeX 内容渲染并显示


<IPython.core.display.Latex object>

- 若只需渲染某些数学表达式，也可以使用 `Math` 类:

In [6]:
from IPython.display import Math  # 导入 Math 用于渲染数学公式

display(Math(r"\dfrac{14}{3}"))   # 以 LaTeX 形式显示分数 14/3


<IPython.core.display.Math object>

&nbsp;
## 3.3 编写封装以简化文本生成

- 上文我们已经加载了预训练 LLM，并搭建好了文本生成功能（如下图所示），它们正是本章余下内容将覆盖的评估流程前两步。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F05_raschka.webp?1" width="500px">

- 为了更方便，我们再封装一层文本生成函数，调用时只需传入模型、tokenizer、提示语和相关设置。

In [7]:
def generate_text_stream_concat(                      # 流式生成并拼接输出的辅助函数
    model, tokenizer, prompt, device, max_new_tokens,
    verbose=False,
):
    input_ids = torch.tensor(                         # 将提示词编码为张量并加 batch 维
        tokenizer.encode(prompt), device=device
    ).unsqueeze(0)

    generated_ids = []                                # 存放生成的 token id
    for token in generate_text_basic_stream_cache(    # 逐 token 流式生成
        model=model,
        token_ids=input_ids,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    ):
        next_token_id = token.squeeze(0)              # 去掉 batch 维
        generated_ids.append(next_token_id.item())    # 收集生成的 token id

        if verbose:                                   # 若需要实时输出
            print(
                tokenizer.decode(next_token_id.tolist()),
                end="",
                flush=True
            )
    return tokenizer.decode(generated_ids)            # 解码全部生成结果为字符串


skip_portion = False                                  # 是否跳过本段生成逻辑

if not skip_portion:                                  # 未跳过则执行生成
    generated_text = generate_text_stream_concat(
        model, tokenizer, prompt, device,
        max_new_tokens=2048,                          # 最多生成 2048 个 token
        verbose=True                                  # 流式打印输出
    )


 To find the value of \( a^2 + b^2 \) given that \( a + b = 3 \) and \( ab = \frac{13}{6} \), we can use the following algebraic identity:

\[
a^2 + b^2 = (a + b)^2 - 2ab
\]

**Step 1:** Substitute the given values into the equation.

\[
a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
\]

**Step 2:** Calculate \( (3)^2 \).

\[
(3)^2 = 9
\]

**Step 3:** Calculate \( 2 \times \frac{13}{6} \).

\[
2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3}
\]

**Step 4:** Subtract the second result from the first.

\[
a^2 + b^2 = 9 - \frac{13}{3}
\]

**Step 5:** Convert 9 to a fraction with a denominator of 3 to perform the subtraction.

\[
9 = \frac{27}{3}
\]

\[
a^2 + b^2 = \frac{27}{3} - \frac{13}{3} = \frac{14}{3}
\]

**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]

&nbsp;
## 3.4 提取最终答案框

- 本节先提取答案框（第 3 步）；下一节会把提取出的答案做标准化处理（第 4 步）。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F06_raschka.webp?1" width="500px">

In [8]:
model_answer = (                       # 存放模型输出（包含解释与最终答案）的多行字符串
r"""... some explanation...
**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]
""")


In [9]:
def get_last_boxed(text):
    # 找到字符串中最后一次出现 "\boxed" 的位置
    boxed_start_idx = text.rfind(r"\boxed")
    if boxed_start_idx == -1:
        return None  # 没有找到则返回 None

    # 定位到 "\boxed" 之后的第一个字符索引
    current_idx = boxed_start_idx + len(r"\boxed")

    # 跳过 "\boxed" 后的空白字符
    while current_idx < len(text) and text[current_idx].isspace():
        current_idx += 1

    # 期望下一个字符是左花括号 "{"
    if current_idx >= len(text) or text[current_idx] != "{":
        return None

    # 解析花括号，支持嵌套
    current_idx += 1
    brace_depth = 1
    content_start_idx = current_idx

    while current_idx < len(text) and brace_depth > 0:
        char = text[current_idx]
        if char == "{":
            brace_depth += 1  # 遇到左括号，深度加一
        elif char == "}":
            brace_depth -= 1  # 遇到右括号，深度减一
        current_idx += 1

    # 如果括号不平衡，返回 None
    if brace_depth != 0:
        return None

    # 提取最外层花括号内的内容并返回
    return text[content_start_idx:current_idx-1]


In [10]:
extracted_answer = get_last_boxed(model_answer)  # 从模型答案中提取最后一个 \boxed{} 内容
print(extracted_answer)                          # 输出提取到的结果


\dfrac{14}{3}


In [11]:
import re  # 正则表达式库

RE_NUMBER = re.compile(               # 匹配整数、分数、浮点、科学计数法的模式
    r"-?(?:\d+/\d+|\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)"
)

def extract_final_candidate(text, fallback="number_then_full"):
    # 默认返回值，当没有匹配时使用
    result = ""

    if text:
        # 优先尝试提取最后一个 \boxed{} 内容
        boxed = get_last_boxed(text.strip())
        if boxed:
            result = boxed.strip().strip("$ ")

        # 若没有 \boxed，按回退策略处理
        elif fallback in ("number_then_full", "number_only"):
            m = RE_NUMBER.findall(text)  # 查找所有数字/分数/科学计数法匹配
            if m:
                result = m[-1]           # 取最后一个匹配的数字
            elif fallback == "number_then_full":
                result = text            # 若无数字且允许，返回全文
    return result                        # 返回提取结果


- 当找不到 boxed 内容时的后备策略：
    - "number_then_full": 优先提取最后一个简单数字，否则返回整段文本。
    - "number_only": 只提取最后一个简单数字，若没有则返回空字符串 `""`
    - "none": 仅提取 boxed 内容，若未找到则返回空字符串 `""`

In [12]:
print(extract_final_candidate(model_answer))  # 打印提取的最终候选答案


\dfrac{14}{3}


In [13]:
print(extract_final_candidate(r"\boxed{ 14/3. }"))  # 提取 \boxed 中的数值并打印结果

14/3.


In [14]:
print(extract_final_candidate("abc < > 14/3 abc"))  # 从字符串中提取最后的数字/分数并打印结果


14/3


In [15]:
print(extract_final_candidate("Text without numbers"))  # 输入无数字时按回退策略打印结果


Text without numbers


&nbsp;
## 3.5 标准化提取出的答案

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F07_raschka.webp?1" width="500px">

- 上一节完成了答案提取（步骤 3），现在我们要对其做标准化处理（对应上一张图的步骤 4）。

In [16]:
LATEX_FIXES = [  # 需要替换或删减的 LaTeX 片段列表（模式, 替换为）
    (r"\\left\s*", ""),
    (r"\\right\s*", ""),
    (r"\\,|\\!|\\;|\\:", ""),
    (r"\\cdot", "*"),
    (r"\u00B7|\u00D7", "*"),
    (r"\\\^\\circ", ""),
    (r"\\dfrac", r"\\frac"),
    (r"\\tfrac", r"\\frac"),
    (r"°", ""),
]

RE_SPECIAL = re.compile(r"<\|[^>]+?\|>")  # 匹配并移除聊天特殊 token，如 <|assistant|>

def normalize_text(text):
    if not text:
        return ""  # 空输入直接返回空串
    text = RE_SPECIAL.sub("", text).strip()  # 去掉特殊 token 并裁剪空白

    # 移除角度符号与标记
    text = re.sub(r"\^\s*\{\s*\\circ\s*\}", "", text)   # ^{\circ}
    text = re.sub(r"\^\s*\\circ", "", text)             # ^\circ
    text = text.replace("°", "")                        # Unicode 角度符

    # 若整串形如 \text{...} 则解包
    match = re.match(r"^\\text\{(?P<x>.+?)\}$", text)
    if match:
        text = match.group("x")

    # 去掉行内/行间数学环境包装 \( \) \[ \]
    text = re.sub(r"\\\(|\\\)|\\\[|\\\]", "", text)

    # 轻量 LaTeX 规范化
    for pat, rep in LATEX_FIXES:
        text = re.sub(pat, rep, text)

    # 处理根号与百分号
    text = text.replace("\\%", "%").replace("$", "").replace("%", "")
    text = re.sub(
        r"\\sqrt\s*\{([^}]*)\}",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )
    text = re.sub(
        r"\\sqrt\s+([^\\\s{}]+)",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )

    # 处理分数形式
    text = re.sub(
        r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )
    text = re.sub(
        r"\\frac\s+([^\s{}]+)\s+([^\s{}]+)",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )

    # 幂次与带分数：^ 换成 **，数字后跟空格分数转为加法
    text = text.replace("^", "**")
    text = re.sub(
        r"(?<=\d)\s+(\d+/\d+)",
        lambda match: "+" + match.group(1),
        text,
    )

    # 千分位逗号移除：1,234 -> 1234
    text = re.sub(
        r"(?<=\d),(?=\d\d\d(\D|$))",
        "",
        text,
    )

    return text.replace("{", "").replace("}", "").strip().lower()  # 去花括号并统一小写


In [17]:
print(normalize_text(extract_final_candidate(model_answer)))  # 提取模型答案中的最终值并做规范化后打印


(14)/(3)


In [18]:
print(normalize_text(r"$\dfrac{14}{3.}$"))  # 将 LaTeX 分数规范化并打印结果


(14)/(3.)


In [19]:
print(normalize_text(r"\text{\[\frac{14}{3}\]}"))  # 规范化 LaTeX 字符串并打印结果


(14)/(3)


In [20]:
print(normalize_text("4/3"))  # 直接规范化普通分数字符串并打印结果


4/3


&nbsp;
## 3.6 验证数学等价性

- 本节实现基础功能：比较模型生成并提取出的答案与数据集中提供的正确答案（ground truth）是否等价，对应流程的第 5 步。
- 下一节（第 6 步）会进一步增强判分逻辑，让答案正确性评估更稳健。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F08_raschka.webp?1" width="500px">

In [21]:
from sympy.parsing import sympy_parser as spp
from sympy.core.sympify import SympifyError

def sympy_parser(expr):
    try:
        return spp.parse_expr(
            expr,
            transformations=(
                # Standard transformations like handling parentheses
                *spp.standard_transformations,

                # Allow omitted multiplication symbols (e.g., "2x" -> 2*x")
                spp.implicit_multiplication_application,
            ),

            # Evaluate during parsing so simple constants simplify (e.g., 2+3 -> 5)
            evaluate=True,
        )
    except (SympifyError, SyntaxError, TypeError, IndexError):
        return None

- 看起来这里的错误处理很多，但这些都是我在评估 500 道 MATH-500 题目时遇到的真实问题，因为模型输出并不总是格式完美。

In [22]:
print(sympy_parser(normalize_text(
    extract_final_candidate(model_answer)
)))

14/3


In [23]:
print(sympy_parser("28/6"))

14/3


In [ ]:
from sympy.parsing import sympy_parser as spp      # 导入 Sympy 解析工具
from sympy.core.sympify import SympifyError        # 解析失败的异常类型

def sympy_parser(expr):
    try:
        return spp.parse_expr(
            expr,
            transformations=(
                # 标准转换（括号处理等）
                *spp.standard_transformations,
                # 允许省略乘号（如 2x 解析为 2*x）
                spp.implicit_multiplication_application,
            ),
            # 解析时直接求值（如 2+3 -> 5）
            evaluate=True,
        )
    except (SympifyError, SyntaxError, TypeError, IndexError):
        return None  # 解析失败返回 None


In [25]:
print(equality_check(                      # 调用比较函数，检查两个表达式是否等价
    normalize_text("13/4."),              # 先对字符串 "13/4." 做规范化
    normalize_text(r"(13)/(4)")           # 再对 "(13)/(4)" 做规范化
))


True


In [26]:
print(equality_check(
    normalize_text("0.5"),
    normalize_text(r"(1)/(2)")
))

True


In [27]:
print(equality_check(
    normalize_text("14/3"),
    normalize_text("15/3")
))

False


In [28]:
print(equality_check(
    normalize_text("(14/3, 2/3)"),
    normalize_text("(14/3, 4/6)")
))

False


&nbsp;
## 3.7 Grading answers

In [ ]:
def split_into_parts(text):
    result = [text]  # 默认把整个字符串作为单一结果

    if text:
        # 如果形如 "(a, b)" 或 "[a, b]" 且内部包含逗号，则尝试按逗号拆分
        if (
            len(text) >= 2
            and text[0] in "([" and text[-1] in ")]"
            and "," in text[1:-1]
        ):
            items = [p.strip() for p in text[1:-1].split(",")]  # 去掉括号后按逗号切分并裁剪空白
            if all(items):  # 确保每个分段非空
                result = items
    else:
        result = []  # 空字符串则返回空列表

    return result


In [30]:
split_into_parts(normalize_text(r"(14/3, 2/3)"))  # 规范化后拆分元组样式字符串，返回 ["14/3", "2/3"]


['14/3', '2/3']

In [ ]:
def grade_answer(pred_text, gt_text):
    result = False  # 默认评判为错误

    # 仅当两侧非空时继续比较
    if pred_text is not None and gt_text is not None:
        gt_parts = split_into_parts(
            normalize_text(gt_text)
        )  # 规范化并拆分标准答案

        pred_parts = split_into_parts(
            normalize_text(pred_text)
        )  # 规范化并拆分模型预测

        # 部分数量需一致且都非空
        if (gt_parts and pred_parts
           and len(gt_parts) == len(pred_parts)):
            result = all(
                equality_check(gt, pred)
                for gt, pred in zip(gt_parts, pred_parts)
            )  # 对应位置逐一检验数学等价

    return result  # 全部匹配则返回 True


In [32]:
grade_answer("14/3", r"\frac{14}{3}")

True

In [33]:
grade_answer(r"(14/3, 2/3)", "(14/3, 4/6)")

True

In [ ]:
# 定义测试用例列表：(名称，模型预测，标准答案，期望布尔结果)
tests = [
        ("check_1", "3/4", r"\frac{3}{4}", True),
        ("check_2", "(3)/(4)", r"3/4", True),
        ("check_3", r"\frac{\sqrt{8}}{2}", "sqrt(2)", True),
        ("check_4", r"\( \frac{1}{2} + \frac{1}{6} \)", "2/3", True),
        ("check_5", "(1, 2)", r"(1,2)", True),
        ("check_6", "(2, 1)", "(1, 2)", False),
        ("check_7", "(1, 2, 3)", "(1, 2)", False),
        ("check_8", "0.5", "1/2", True),
        ("check_9", "0.3333333333", "1/3", False),
        ("check_10", "1,234/2", "617", True),
        ("check_11", r"\text{2/3}", "2/3", True),
        ("check_12", "50%", "1/2", False),
        ("check_13", r"2\cdot 3/4", "3/2", True),
        ("check_14", r"90^\circ", "90", True),
        ("check_15", r"\left(\frac{3}{4}\right)", "3/4", True),
    ]


def run_demos_table(tests):
    header = ("Test", "Expect", "Got", "Status")  # 表头
    rows = []
    for name, pred, gtruth, expect in tests:
        got = grade_answer(pred, gtruth)          # 调用评分函数
        status = "PASS" if got == expect else "FAIL"  # 根据期望与实际决定状态
        rows.append((name, str(expect), str(got), status))

    data = [header] + rows
    
    # 计算每列最大宽度，用于对齐输出
    col_widths = [
        max(len(row[i]) for row in data)
        for i in range(len(header))
    ]

    # 按行打印表格
    for row in data:
        line = " | ".join(
            row[i].ljust(col_widths[i])
            for i in range(len(header))
        )
        print(line)

    # 打印通过用例的数量总结
    passed = sum(r[3] == "PASS" for r in rows)
    print(f"\nPassed {passed}/{len(rows)}")


In [35]:
run_demos_table(tests)

Test     | Expect | Got   | Status
check_1  | True   | True  | PASS  
check_2  | True   | True  | PASS  
check_3  | True   | True  | PASS  
check_4  | True   | True  | PASS  
check_5  | True   | True  | PASS  
check_6  | False  | False | PASS  
check_7  | False  | False | PASS  
check_8  | True   | True  | PASS  
check_9  | False  | False | PASS  
check_10 | True   | True  | PASS  
check_11 | True   | True  | PASS  
check_12 | False  | False | PASS  
check_13 | True   | True  | PASS  
check_14 | True   | True  | PASS  
check_15 | True   | True  | PASS  

Passed 15/15


&nbsp;
## 3.8 加载评测数据集

- 上一节实现了基础的评估流水线。
- 本节加载要评估的数据集（步骤 7），下一节会在该数据集上应用这套流程来评估模型（步骤 8）。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F09_raschka.webp?1" width="500px">

- 该数据集是通过下述脚本从 [HuggingFaceH4/MATH-500](https://huggingface.co/datasets/HuggingFaceH4/MATH-500)  仓库下载并预处理的，该脚本依赖  [`datasets`](https://huggingface.co/docs/datasets/en/index) 包（无需实际运行，仅供参考）：

```python
from datasets import load_dataset
import json

dset = load_dataset("HuggingFaceH4/MATH-500", split="test")

math_data = dset.to_list()
with open("math500_test.json", "w", encoding="utf-8") as f:
    json.dump(math_data, f, ensure_ascii=False, indent=2)
```

In [36]:
import json                      # 处理 JSON 数据
import requests                  # 发送 HTTP 请求获取远程文件

local_path = Path("math500_test.json")  # 本地文件路径
url = (                                   # 远程文件 URL
    "https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/"
    "main/ch03/01_main-chapter-code/math500_test.json"
)

if local_path.exists():                   # 若本地已存在文件，直接读取
    with local_path.open("r", encoding="utf-8") as f:
        math_data = json.load(f)
else:                                     # 否则从远程下载
    r = requests.get(url, timeout=30)
    r.raise_for_status()                  # 若状态码异常则抛出错误
    math_data = r.json()                  # 解析为 Python 对象

print("Number of entries:", len(math_data))  # 打印条目数量


Number of entries: 500


In [37]:
from pprint import pprint  # 导入 pprint 以更易读的格式打印对象
pprint(math_data[0])       # 打印数据集中第一个条目，查看其结构


{'answer': '\\left( 3, \\frac{\\pi}{2} \\right)',
 'level': 2,
 'problem': 'Convert the point $(0,3)$ in rectangular coordinates to polar '
            'coordinates.  Enter your answer in the form $(r,\\theta),$ where '
            '$r > 0$ and $0 \\le \\theta < 2 \\pi.$',
 'solution': 'We have that $r = \\sqrt{0^2 + 3^2} = 3.$  Also, if we draw the '
             'line connecting the origin and $(0,3),$ this line makes an angle '
             'of $\\frac{\\pi}{2}$ with the positive $x$-axis.\n'
             '\n'
             '[asy]\n'
             'unitsize(0.8 cm);\n'
             '\n'
             'draw((-0.5,0)--(3.5,0));\n'
             'draw((0,-0.5)--(0,3.5));\n'
             'draw(arc((0,0),3,0,90),red,Arrow(6));\n'
             '\n'
             'dot((0,3), red);\n'
             'label("$(0,3)$", (0,3), W);\n'
             'dot((3,0), red);\n'
             '[/asy]\n'
             '\n'
             'Therefore, the polar coordinates are $\\boxed{\\left( 3, '
             '\\frac

&nbsp;
## 3.9 评估模型

- 上一节已经加载了数据集；现在可以把前面搭好的评估流程应用到该数据集上，对模型进行评估（对应步骤 7）。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F10_raschka.webp?1" width="500px">

In [ ]:
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"         # 角色与任务说明
        "Answer the question and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"                          # 要求用 \boxed{} 给出最终答案
        f"Question:\n{prompt}\n\nAnswer:"             # 插入具体题目并提示开始作答
    )
    return template                                   # 返回完整提示词模版


In [39]:
prompt = (  # 与本章开头相同的数学题提示
    r"If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?"
)
prompt_fmt = render_prompt(prompt)  # 使用模板渲染成完整提示
print(prompt_fmt)                   # 打印渲染后的提示词


You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
If $a+b=3$ and $ab=\tfrac{13}{6}$, what is the value of $a^2+b^2$?

Answer:


In [40]:
generated_text = generate_text_stream_concat(   # 调用流式生成函数
    model, tokenizer, prompt_fmt, device,       # 传入模型、分词器、格式化后的提示词和设备
    max_new_tokens=2048,                        # 最多生成 2048 个 token
    verbose=True                                # 实时打印生成内容
)


 \boxed{10}

In [47]:
# 下方是替代提示模板：用 “Problem” 取代 “Question”
"""
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "Solve the problem and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"
        f"Problem:\n{prompt}\n\nAnswer:"
    )
    return template
"""

# 该改动会影响 MATH-500 表现：
# Base 模型 (mps)：准确率约从 20% 提升到 40%
# Reasoning 模型 (mps)：准确率约从 90% 降到 60%


'\ndef render_prompt(prompt):\n    template = (\n        "You are a helpful math assistant.\n"\n        "Solve the problem and write the final result on a new line as:\n"\n        "\\boxed{ANSWER}\n\n"\n        f"Problem:\n{prompt}\n\nAnswer:"\n    )\n    return template\n'

In [48]:
# 另一个选项：不使用提示模板，直接返回原始题目
"""
def render_prompt(prompt):
    return prompt
"""

# 这样也会显著影响 MATH-500 结果：
# Base 模型 (mps)：准确率约从 20% 提升到 70%
# Reasoning 模型 (mps)：准确率约从 90% 降到 50%


'\ndef render_prompt(prompt):\n    return prompt\n'

In [ ]:
def mini_eval_demo(model, tokenizer, device):
    ex = {  # 测试样例，包含题目与标准答案
        "problem": "Compute 1/2 + 1/6.",
        "answer": "2/3"
    }
    prompt = render_prompt(ex["problem"])         # 1) 应用提示模板
    gen_text = generate_text_stream_concat(       # 2) 生成模型回复
        model, tokenizer, prompt, device,
        max_new_tokens=64,
    )
    pred_answer = extract_final_candidate(gen_text)  # 3) 提取并规范化模型答案
    is_correct = grade_answer(                       # 4) 评分：判断与标准答案是否等价
        pred_answer, ex["answer"]
    )
    print(f"Device: {device}")
    print(f"Prediction: {pred_answer}")
    print(f"Ground truth: {ex['answer']}")
    print(f"Correct: {is_correct}")


In [50]:
mini_eval_demo(model, tokenizer, device)  # 运行快速评估，打印设备、预测答案、标准答案以及是否正确

Device: cuda
Prediction: 1/3
Ground truth: 2/3
Correct: False


In [54]:
import time  # 时间模块，用于计时

# 计算剩余时间的辅助函数
def eta_progress_message(
    processed,
    total,
    start_time,
    show_eta=False,
    label="Progress",
):
    progress = f"{label}: {processed}/{total}"  # 当前进度文本
    if not show_eta or processed <= 0:
        return progress  # 若不显示 ETA 或尚未开始，直接返回进度

    elapsed = time.time() - start_time          # 已用时间
    if elapsed <= 0:
        return progress

    remaining = max(total - processed, 0)       # 剩余条数

    if processed:
        avg_time = elapsed / processed          # 平均每条耗时
        eta_seconds = avg_time * remaining      # 估计剩余秒数
    else:
        eta_seconds = 0

    eta_seconds = max(int(round(eta_seconds)), 0)
    minutes, rem_seconds = divmod(eta_seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        eta = f"{hours}h {minutes:02d}m {rem_seconds:02d}s"
    elif minutes:
        eta = f"{minutes:02d}m {rem_seconds:02d}s"
    else:
        eta = f"{rem_seconds:02d}s"

    return f"{progress} | ETA: {eta}"           # 返回带 ETA 的进度字符串


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=512,
    verbose=False,
):

    if out_path is None:
        dev_name = str(device).replace(":", "-")     # 设备名替换冒号以兼容 Windows 文件名
        out_path = Path(f"math500-{dev_name}.jsonl") # 默认输出路径

    num_examples = len(math_data)                    # 总样本数
    num_correct = 0                                  # 累计正确数
    start_time = time.time()                         # 记录开始时间

    with open(out_path, "w", encoding="utf-8") as f:  # 保存详细日志，便于检查
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"])    # 1. 构造提示词
            gen_text = generate_text_stream_concat(   # 2. 生成模型回复
                model, tokenizer, prompt, device,
                max_new_tokens=max_new_tokens,
                verbose=verbose,
            )

            extracted = extract_final_candidate(      # 3. 提取与规范化答案
                gen_text
            )
            is_correct = grade_answer(                # 4. 判分：与标准答案比对
                extracted, row["answer"]
            )
            num_correct += int(is_correct)            # 累计正确数

            record = {                                # 保存当前样本的日志记录
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            progress_msg = eta_progress_message(      # 构造带 ETA 的进度提示
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)  # 实时更新单行进度
            if verbose:  # 若需要详细输出，打印分隔信息
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )

    # 结束后打印汇总
    seconds_elapsed = time.time() - start_time
    acc = num_correct / num_examples if num_examples else 0.0
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")
    print(f"Total time: {seconds_elapsed/60:.1f} min")
    print(f"Logs written to: {out_path}")
    return num_correct, num_examples, acc          # 返回正确数、总数、准确率


- 为了演示并控制运行时间，这里只评估 10 个样例。

In [56]:
print("Model:", WHICH_MODEL)                     # 输出使用的模型类型
print("Device:", device)                         # 输出当前设备（CPU/GPU）
num_correct, num_examples, acc = evaluate_math500_stream(  # 跑前 10 道题的评测
    model, tokenizer, device, 
    math_data=math_data[:10],                    # 仅取前 10 个样本做快速测试
    max_new_tokens=2048,                         # 生成上限 token 数
    verbose=False                                # 关闭详细日志输出
)


Model: base
Device: cuda
MATH-500: 10/10 | ETA: 00s
Accuracy: 30.0% (3/10)
Total time: 0.6 min
Logs written to: math500-cuda.jsonl


| Mode      | Device | Accuracy | MATH-500 size | Time                  |
|-----------|--------|----------|---------------|-----------------------|
| Base      | CPU    | 30%      | 10            | 0.7 min (Mac Mini M4) |
| Base      | MPS    | 20%      | 10            | 0.4 min (Mac Mini M4) |
| Base      | CUDA   | 30%      | 10            | 0.2 min (DGX Spark)   |
| Base      | XPU    | 30%      | 10            | 1.2 min (Intel)       |
| Reasoning | CPU    | 90%      | 10            | 9.5 min (Mac Mini M4) |
| Reasoning | MPS    | 80%      | 10            | 3.8 min (Mac Mini M4) |
| Reasoning | CUDA   | 90%      | 10            | 3.7 min (DGX Spark)   |
| Reasoning | XPU    | 70%      | 10            | 8.5 min (Intel)       |


| Mode      | Device | Accuracy | MATH-500 size    | Time                   |
|-----------|--------|----------|------------------|------------------------|
| Base      | CUDA   | 15.6%    | 500              | 10.0 min (DGX Spark)   |
| Reasoning | CUDA   | 50.8%    | 500              | 182.2 min (DGX Spark)  |

- 请注意，这些数值是在 PyTorch 2.8 上得到的，换成其它版本可能会有所不同。

- 上表列出了不同配置下的准确率。
- 这里的 “GPU” 指 NVIDIA（cuda）显卡；MPS 指 Apple Silicon M4。
- 推理版模型更慢，是因为它输出的文本更长。
- 虽然 Qwen3-Base 是基础预训练模型，且官方建议不用 chat 模板，但把 `tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)` 改成 `tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path, apply_chat_template=True)` 能显著提升 MATH-500 表现（可达 80%）。需要指出，MATH-500 测试集是否被用于训练尚不明确；在 LLM 时代，通常假定互联网上的数据都有可能进入训练集(参见 [here](https://github.com/rasbt/LLMs-from-scratch/pull/828#issuecomment-3324829736))
- 要跑完整 500 道 MATH-500 题，只需把 `evaluate_math500_stream` 中的 `math_data=math_data[:10],`  改成 `math_data=math_data,`
- T附加资料中包含批处理版脚本，可大幅提升评估吞吐(见 [../02_math500-verifier-scripts/README.md](../02_math500-verifier-scripts/README.md))在 H100 上，batch size=128 时，基础模型 3.3 分钟就能评完、推理版大约 14.6 分钟。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F11_raschka.webp?1" width="500px">

- 可直接使用 [../02_math500-verifier-scripts/evaluate_math500.py](../02_math500-verifier-scripts/evaluate_math500.py) 脚本，它可在命令行单独运行 MATH-500 评估流程；更多用法见([../02_math500-verifier-scripts/README.md](../02_math500-verifier-scripts/README.md) )
- The [../02_math500-verifier-scripts/evaluate_math500_batched.py](../02_math500-verifier-scripts/evaluate_math500_batched.py) 会以批处理模式执行本章代码：
  - 即一次前向传播会处理多个样本，提升评估速度，但需要更多内存；
  - 在 H100 GPU 上，batch size=128 时，评测 500 个样本的基础模型耗时可从 13.3 分钟降到 3.3 分钟；
  - 推理模型同样可从 185.4 分钟降到 14.6 分钟；
  - H100 仅作示例，该脚本也兼容其他 GPU（甚至 CPU）。

&nbsp;
## 总结

- 本节没有代码。